# NEUROLOG Replication: Neuro-Symbolic Training with WMC Semantic Loss

This notebook runs the full **3-way ablation study** from the NEUROLOG architecture (Tsamoura et al., AAAI 2021):

| Strategy | Description |
|---|---|
| **Pure Neural** | Standard cross-entropy, no symbolic feedback |
| **NGA** | Neural-Guided Abduction (single best correction path) |
| **WMC** | Weighted Model Counting via d-DNNF circuits (full probabilistic semantic loss) |

All three share the same initial weights for fair comparison.

**Runtime:** Set to **T4 GPU** (Runtime > Change runtime type).

## 1. Setup: Clone Repo & Install Dependencies

In [ ]:
import os

REPO_URL = 'https://github.com/konstantinoslamp/NeuroSymbolic-Network-for-Handwritten-Symbol-Understanding.git'
REPO_DIR = '/content/neurosymbolic_mvp'

if os.path.exists(REPO_DIR):
    print('Repo exists, pulling latest...')
    os.chdir(REPO_DIR)
    !git pull
else:
    !git clone {REPO_URL} {REPO_DIR}
    os.chdir(REPO_DIR)

print(f'Working directory: {os.getcwd()}')

In [ ]:
!pip install -q clingo numpy tqdm matplotlib Pillow

In [ ]:
# Verify clingo works
import clingo
print(f'clingo version: {clingo.__version__}')
print('All dependencies installed.')

## 2. Download MNIST

In [ ]:
import urllib.request
import os

mnist_path = os.path.join(REPO_DIR, 'src', 'data', 'mnist.npz')
os.makedirs(os.path.dirname(mnist_path), exist_ok=True)

if not os.path.exists(mnist_path):
    print('Downloading MNIST...')
    url = 'https://storage.googleapis.com/tensorflow/tf-keras-datasets/mnist.npz'
    urllib.request.urlretrieve(url, mnist_path)
    print('Done.')
else:
    print('MNIST already present.')

## 3. Verify Symbolic KB (clingo)

In [ ]:
import sys
sys.path.insert(0, REPO_DIR)

from src.symbolic.knowledge_base import KnowledgeBase

kb = KnowledgeBase()
print('Deduction: 3 + 5 =', kb.deduce(3, '+', 5))
print('Deduction: 7 - 2 =', kb.deduce(7, '-', 2))
print('Abduction (target=8):', kb.abduce(8.0)[:5], '...')
print('Symbolic KB fully functional.')

## 4. Run Full Ablation Study (Pure Neural vs NGA vs WMC)

In [ ]:
# Hyperparameters - adjust these as needed
EPOCHS = 10
TRAIN_SAMPLES = 2000
TEST_SAMPLES = 500
BATCH_SIZE = 32
LEARNING_RATE = 0.01

In [ ]:
import numpy as np
import time
import json

from src.data.expression_dataset import ExpressionDataset
from src.neural.digit_recognizer import DigitRecognizer
from src.symbolic.symbolic_interface import ArithmeticSymbolicModule
from src.evaluation.ablation_studies import AblationRunner, ABLATION_CONFIGS, AblationConfig

# Build datasets
print('Building datasets...')
train_dataset = ExpressionDataset(num_samples=TRAIN_SAMPLES, split='train', invalid_ratio=0.05)
test_dataset  = ExpressionDataset(num_samples=TEST_SAMPLES, split='test', invalid_ratio=0.05)
print(f'Train: {len(train_dataset)} samples  |  Test: {len(test_dataset)} samples')

In [ ]:
# Configure ablation runs
configs = {
    name: AblationConfig(
        name=cfg.name,
        use_symbolic=cfg.use_symbolic,
        abduction_strategy=cfg.abduction_strategy,
        epochs=EPOCHS,
        learning_rate=LEARNING_RATE,
        batch_size=BATCH_SIZE,
    )
    for name, cfg in ABLATION_CONFIGS.items()
}

runner = AblationRunner(
    neural_module_factory=DigitRecognizer,
    symbolic_module_factory=ArithmeticSymbolicModule,
    train_dataset=train_dataset,
    test_dataset=test_dataset,
)

print(f'Running {len(configs)} ablations x {EPOCHS} epochs each...')
print('This may take 10-30 minutes depending on sample size.')
t0 = time.time()
results = runner.run_all(configs=configs)
elapsed = time.time() - t0
print(f'\nAll ablations complete in {elapsed:.1f}s')

## 5. Print Comparison Table

In [ ]:
AblationRunner.print_comparison(results)

## 6. Plot Training Curves

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = {'pure_neural': '#e74c3c', 'nga': '#3498db', 'wmc': '#2ecc71'}
labels = {'pure_neural': 'Pure Neural', 'nga': 'NGA', 'wmc': 'WMC'}

# Loss curves
ax = axes[0]
for name, res in results.items():
    history = res['training_history']
    epochs_list = [h['epoch'] for h in history]
    losses = [h['avg_loss'] for h in history]
    ax.plot(epochs_list, losses, color=colors.get(name, 'gray'),
            label=labels.get(name, name), linewidth=2, marker='o', markersize=4)
ax.set_xlabel('Epoch')
ax.set_ylabel('Training Loss')
ax.set_title('Training Loss per Epoch')
ax.legend()
ax.grid(True, alpha=0.3)

# Accuracy curves
ax = axes[1]
for name, res in results.items():
    history = res['training_history']
    epochs_list = [h['epoch'] for h in history]
    accs = [h['accuracy'] for h in history]
    ax.plot(epochs_list, accs, color=colors.get(name, 'gray'),
            label=labels.get(name, name), linewidth=2, marker='o', markersize=4)
ax.set_xlabel('Epoch')
ax.set_ylabel('Training Accuracy')
ax.set_title('Training Accuracy per Epoch')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('ablation_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: ablation_curves.png')

## 7. Plot Evaluation Bar Chart

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

metric_names = ['Digit Acc', 'Operator Acc', 'Expression Acc', 'Result Acc']
extractors = [
    lambda r: r['evaluation']['per_class']['overall_digit_accuracy'],
    lambda r: r['evaluation']['per_class']['overall_operator_accuracy'],
    lambda r: r['evaluation']['expression']['expression_accuracy'],
    lambda r: r['evaluation']['result']['result_accuracy'],
]

x = np.arange(len(metric_names))
width = 0.25

for i, (name, res) in enumerate(results.items()):
    values = []
    for ext in extractors:
        try:
            values.append(ext(res))
        except (KeyError, TypeError):
            values.append(0.0)
    ax.bar(x + i * width, values, width, label=labels.get(name, name),
           color=colors.get(name, 'gray'), alpha=0.85)

ax.set_ylabel('Accuracy')
ax.set_title('Ablation Study: Pure Neural vs NGA vs WMC')
ax.set_xticks(x + width)
ax.set_xticklabels(metric_names)
ax.legend()
ax.set_ylim(0, 1.05)
ax.grid(True, alpha=0.3, axis='y')

# Add value labels
for container in ax.containers:
    ax.bar_label(container, fmt='%.2f', fontsize=8, padding=2)

plt.tight_layout()
plt.savefig('ablation_barplot.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: ablation_barplot.png')

## 8. Detailed Evaluation Report

In [ ]:
from src.evaluation.metrics import EvaluationSuite

# Print detailed report for the WMC strategy (our main result)
if 'wmc' in results:
    print('=== WMC (Weighted Model Counting) - Detailed Report ===')
    EvaluationSuite.print_report(results['wmc']['evaluation'])
else:
    # Print whatever is available
    for name, res in results.items():
        print(f'\n=== {res["config"]} - Detailed Report ===')
        EvaluationSuite.print_report(res['evaluation'])

## 9. Save Results

In [ ]:
import json

def convert_numpy(obj):
    """Recursively convert numpy types for JSON serialization."""
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, np.floating):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, dict):
        return {k: convert_numpy(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [convert_numpy(v) for v in obj]
    return obj

os.makedirs('results', exist_ok=True)
output_path = 'results/neurolog_ablation_results.json'

with open(output_path, 'w') as f:
    json.dump(convert_numpy(results), f, indent=2)

print(f'Results saved to: {output_path}')
print(f'File size: {os.path.getsize(output_path) / 1024:.1f} KB')

In [ ]:
# Download results + plots
try:
    from google.colab import files
    files.download(output_path)
    files.download('ablation_curves.png')
    files.download('ablation_barplot.png')
    print('Downloads triggered.')
except ImportError:
    print('Not running in Colab - files saved locally.')

## 10. Summary Table (Copy for Paper)

Run this cell to get a clean LaTeX-ready table.

In [ ]:
print('\n' + '='*80)
print('  RESULTS SUMMARY (Paper-Ready)')
print('='*80)

header = f'{"Strategy":<25} {"Digit":>8} {"Operator":>10} {"Expr":>8} {"Result":>8} {"ECE":>8} {"Time(s)":>8}'
print(header)
print('-'*80)

for name, res in results.items():
    ev = res['evaluation']
    try:
        row = (
            f'{res["config"]:<25} '
            f'{ev["per_class"]["overall_digit_accuracy"]:>8.3f} '
            f'{ev["per_class"]["overall_operator_accuracy"]:>10.3f} '
            f'{ev["expression"]["expression_accuracy"]:>8.3f} '
            f'{ev["result"]["result_accuracy"]:>8.3f} '
            f'{ev["calibration"]["ece"]:>8.4f} '
            f'{res["train_time_seconds"]:>8.1f}'
        )
        print(row)
    except (KeyError, TypeError) as e:
        print(f'{res["config"]:<25} ERROR: {e}')

print('='*80)
print()
print('LaTeX snippet:')
print(r'\begin{table}[h]')
print(r'\centering')
print(r'\begin{tabular}{lccccc}')
print(r'\toprule')
print(r'Strategy & Digit Acc & Op Acc & Expr Acc & Result Acc & ECE \\')
print(r'\midrule')

for name, res in results.items():
    ev = res['evaluation']
    try:
        print(
            f'{res["config"]} & '
            f'{ev["per_class"]["overall_digit_accuracy"]:.3f} & '
            f'{ev["per_class"]["overall_operator_accuracy"]:.3f} & '
            f'{ev["expression"]["expression_accuracy"]:.3f} & '
            f'{ev["result"]["result_accuracy"]:.3f} & '
            f'{ev["calibration"]["ece"]:.4f} \\\\'
        )
    except (KeyError, TypeError):
        pass

print(r'\bottomrule')
print(r'\end{tabular}')
print(r'\caption{Ablation study: Pure Neural vs NGA vs WMC on MATH(3).}')
print(r'\label{tab:ablation}')
print(r'\end{table}')